# Chatper 2: Working with text data

Verify the packages are available or which version are available using the following code

In [1]:
from importlib.metadata import version

In [2]:
print(f"Torch version: {version("torch")}")
print(f"Tiktoken version: {version("tiktoken")}")

Torch version: 2.10.0
Tiktoken version: 0.12.0


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

### 2.1 Understanding word embeddings

- There are many forms of embeddings; we focus on text embeddings in this book

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLMs work with embeddings in high-dimensional spaces (i.e., thousands of dimensions)
- Since we can't visualize such high-dimensional spaces (we humans think in 1, 2, or 3 dimensions), the figure below illustrates a 2-dimensional embedding space

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

### 2.2 Tokenizing text

In [3]:
# Download sample data

import os
import requests

In [4]:
!ls

chapter02.ipynb the-verdict.txt


In [5]:
file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

In [6]:
!ls

chapter02.ipynb the-verdict.txt


In [7]:
with open(file_path, "r", encoding="utf-8") as file:
    raw_text = file.read()

print(f"Total number of characters: {len(raw_text)}")
print(f"Sample text: {raw_text[:99]}")

Total number of characters: 20479
Sample text: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
- The following regular expression will split on whitespaces

In [8]:
import re

text = "Hello, world. This, is a test."

# Splits on whitespaces
result = re.split(r"(\s)", text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [9]:
# Strip white spaces

result = [item for item in result if item.strip()]
print(result)

['Hello,', 'world.', 'This,', 'is', 'a', 'test.']


- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on

`re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)`

means:

- a group `(...)` capturing delimiters, so split keeps separators.
- alternation `|` between:
    - any one char in character class `[,.:;?_!"()\\']` (comma, dot, colon, semicolon, question mark, underscore, exclamation, double-quote, open/close paren, single quote)
    - literal `--`
    - `\\s` (any whitespace: space, tab, newline, etc.)
- used with `re.split` to split text on punctuation / double hyphen / whitespace while optionally keeping separators in result.

In [10]:
text = "Hello, world. Is this -- a test?"

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [11]:
print(len(preprocessed))

4690


### 2.3 Converting tokens into token IDS

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [13]:
vocab = {
    token:integer 
    for integer, token in enumerate(all_words)
}


In [14]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [15]:
class SimpleTokenizerV1:
    """Simple tokenizer"""

    def __init__(self, vocab: dict):
        self.str_to_int = vocab
        self.int_to_str = {
            i:s for s, i in vocab.items()
        }

    def encode(self, text):
        """Encode text"""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    

    def decode(self, ids: list[int]):
        """Decode ids"""
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

In [16]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
It's the last he painted, you know," 
Mrs. Gisburn said with pardonable pride.
"""

ids = tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
tokenizer.decode(ids)

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [18]:
tokenizer.decode(tokenizer.encode(text))

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

### 2.4 Adding special context tokens

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">